# Clean and preprocess Kordatos

In [5]:
import pandas as pd
import re
import string
from tqdm import tqdm


In [2]:
from bs4 import BeautifulSoup
import glob
import os

### HOCR FILES

In [3]:
# Define the folder containing the files
data_folder = "KORDATOS_JSON"

# Get a list of all .hocr files in the folder
hocr_files = [file for file in os.listdir(data_folder) if file.endswith(".hocr")]

# Initialize an empty list to store page data
data = []

# Process each .hocr file
for file in hocr_files:
    # Extract the page number from the file name
    page_number = int(file.split('_')[-1].split('.')[0])  # Assumes page number is at the end of the file name
    
    # Read the content of the HOCR file
    with open(os.path.join(data_folder, file), 'r', encoding='utf-8') as f:
        hocr_content = f.read()
    
    # Parse the content with BeautifulSoup
    soup = BeautifulSoup(hocr_content, 'html.parser')
    
    # Extract text content
    text = soup.get_text(separator=' ')  # Use separator=' ' to maintain spacing
    
    # Append the data to the list
    data.append({"Page_Number": page_number, "Text": text})



In [4]:
# Create a pandas DataFrame
df = pd.DataFrame(data)

# Sort by page number for consistency
df = df.sort_values(by="Page_Number").reset_index(drop=True)



In [11]:
# Remove Παροραματα from Page 320 and all pages after
df = df[df['Page_Number'] < 263]


In [12]:
# Remove initial pages with titles and other useless information 
df = df[~df['Page_Number'].isin([1,2,3,4,5,6,7])]# Remove initial pages with titles and other useless information 


In [13]:
# reset dataframe index
df = df.reset_index(drop=True)

# renumber actual book pages starting from 7
df['Page_Number'] = range(7, 7 + len(df))


In [37]:
df.shape

(262, 2)

### Normalize dates

In [17]:
def remove_fullstop_from_dates(text):
    # Refined regex pattern to match dates correctly and remove the full stop
    date_pattern = r'(\b\d{1,2}\s+[Α-Ωα-ω]+?\s+\d{4})\.'  # Matches dates like "10 Φεβρουαρίου 1921."
    cleaned_text = re.sub(date_pattern, r'\1', text)
    return cleaned_text

# Example usage
text = "Ημερομηνία είναι 10 Φεβρουαρίου 1921. Τίτλος: Επεξεργασία Κειμένων."
cleaned_text = remove_fullstop_from_dates(text)
print(cleaned_text)

Ημερομηνία είναι 10 Φεβρουαρίου 1921. Τίτλος: Επεξεργασία Κειμένων.


In [18]:
df['Text'] = df['Text'].str.replace(' 10  Φεβρουαρίου  1921. » ', ' 10 Φεβρουαρίου 1921» .', regex=False)

#### Now we  remove footnotes

In [20]:
# in Kordatos footnotes are clearly marked with numbers
def split_footnotes(text):
    """
    Separates main text from footnotes.
    Assumes that the first footnote marker (e.g. 1., 2.)
    begins the footnote section and that footnotes continue
    until the end of the page.
    """
    if not isinstance(text, str):
        return pd.Series([text, None])

    # Footnote marker: 1.  2.  3. etc.
    match = re.search(r'(?<!\d)\b\d{1,2}\.\s+', text)

    if match:
        main_text = text[:match.start()].strip()
        footnotes = text[match.start():].strip()
    else:
        main_text = text.strip()
        footnotes = None

    return pd.Series([main_text, footnotes])

In [21]:
df[['Text_no_foot', 'footnotes']] = df['Text'].apply(split_footnotes)

### Remove OCR artefacts

In [ ]:
results = []

for _, row in df.iterrows():

    page = row["Page_Number"]
    text = str(row["Text_no_foot"])

    # get tokens with Unicode letters/marks 
    for m in re.finditer(r"\S+", text):

        token = m.group()
        start = m.start()

        # at least one letter 
        has_letter = any(ch.isalpha() for ch in token)

        # ALL CAPS?
        is_all_caps = (
            has_letter
            and token.upper() == token
            and token.lower() != token
        )

        if is_all_caps:

            context = text[
                max(0, start - 80):
                min(len(text), m.end() + 120)
            ]

            results.append({
                "Page": page,
                "All_caps_token": token,
                "Char_Position": start,
                "Relative_Position": round(start / len(text), 3) if len(text) else None,
                "Context": context
            })

all_caps = pd.DataFrame(results)

all_caps = all_caps.sort_values(
    ["Page", "Char_Position"]
).reset_index(drop=True)

display(all_caps)


caps_frequency = (
    all_caps["All_caps_token"]
    .value_counts()
    .reset_index()
)

caps_frequency.columns = [
    "All_caps_token",
    "Frequency"
]

In [201]:
caps_frequency = (
    all_caps["All_caps_token"]
    .value_counts()
    .reset_index()
)

caps_frequency.columns = [
    "All_caps_token",
    "Frequency"
]



In [27]:
display(caps_frequency)

In [204]:
# not suspicious all caps instances

exact_artifacts = [
    "OCR",
        
]

for artifact in exact_artifacts:
    df["Text_no_foot"] = (
        df["Text_no_foot"]
        .str.replace(artifact, "", regex=False)
    )

In [205]:
instances = all_caps[
    all_caps["All_caps_token"] == "Ζ»"
]

display(instances)

,Page,All_caps_token,Char_Position,Relative_Position,Context
140,64,Ζ»,1834,0.998,φές Μητροπολιτῶν . ) 64 Γράφτηκε : « Κατὰ μῆνα Μάρτιον 1819. Ἐπινεμήσεως Ζ»


### Check dataframe

In [ ]:
# no need for the footnotes text anymore

In [209]:
df.columns

Index(['Page_Number', 'Text', 'Text_no_foot'], dtype='str')

In [210]:
# Check if there are any NaN values in the 'body' column
has_nans_in_text = df['Text_no_foot'].isnull().any()
# Output the result to check if there are NaNs
print(has_nans_in_text)  # This should print True if there are any NaNs

True


In [211]:
# Identify rows with NaN values in the 'text_no_poem' column
rows_with_nans = df[df['Text_no_foot'].isnull()]

# Display the rows with NaN values
print(rows_with_nans)

     Page_Number Text Text_no_foot
94           101  NaN          NaN
113          120  NaN          NaN
203          210  NaN          NaN


In [212]:
df = df.dropna(subset=['Text_no_foot']).reset_index(drop=True)

## a little cleaning

In [219]:
def clean_text(text):
    """Conservative cleaning for OCR word-count preparation."""

    if not isinstance(text, str) or text.strip() == "":
        return text

    # 1. unite hyphen separated words
    # π.χ. πα- τριαρχείου -> πατριαρχείου
    text = re.sub(
        r'([^\W\d_]+)-\s+([^\W\d_]+)',
        r'\1\2',
        text,
        flags=re.UNICODE
    )

    # 2. remove space before basic punct 
    # π.χ. "πόλιν ." -> "πόλιν."
    text = re.sub(
        r'\s+([.,;:!?··])',
        r'\1',
        text
    )

    # 3. multiple whitespaces -> one space 
    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    # 4. clean start/end
    text = text.strip()

    return text

In [220]:
df["Text_clean"] = (
    df["Text_no_foot"]
    .apply(clean_text)
)

In [221]:
#display the texts better
pd.set_option('display.width', 50)        
pd.set_option('display.max_colwidth', 50)

In [2]:
df.head()

### Fix worsds with middle dot

In [223]:
# Define the function to find words split by a middle dot
def find_middle_dot_splits(text):
    """
    Find words occurring on either side of a middle dot.
    Allows zero or more spaces around the dot.
    """
    if not isinstance(text, str):
        return []

    pattern = r'(\w+)\s*·\s*(\w+)'



In [224]:
for text in df['Text_clean'].dropna():
    if '·' in text:
        print("FOUND:", repr(text[text.index('·')-30:text.index('·')+30]))
        break

FOUND: 'ειακὰ καὶ τὰ ᾿Αθεϊκὰ τοῦ Βόλου· τώρα ἔχουμε καὶ τὴ « Δίκη τῶ'


In [225]:
# check what kind of character is that middle dot
from collections import Counter
import unicodedata

chars = Counter(
    char
    for text in df['Text_clean'].dropna()
    for char in text
    if not char.isalnum() and not char.isspace()
)

for char, count in chars.most_common():
    print(
        repr(char),
        f"U+{ord(char):04X}",
        unicodedata.name(char, "UNKNOWN"),
        count
    )

',' U+002C COMMA 4413
'.' U+002E FULL STOP 3737
'᾿' U+1FBF GREEK PSILI 942
'«' U+00AB LEFT-POINTING DOUBLE ANGLE QUOTATION MARK 561
'»' U+00BB RIGHT-POINTING DOUBLE ANGLE QUOTATION MARK 525
')' U+0029 RIGHT PARENTHESIS 391
'(' U+0028 LEFT PARENTHESIS 357
':' U+003A COLON 281
'·' U+00B7 MIDDLE DOT 156
'–' U+2013 EN DASH 123
';' U+003B SEMICOLON 114
'"' U+0022 QUOTATION MARK 71
'᾽' U+1FBD GREEK KORONIS 62
'!' U+0021 EXCLAMATION MARK 60
'-' U+002D HYPHEN-MINUS 29
'῎' U+1FCE GREEK PSILI AND OXIA 18
'=' U+003D EQUALS SIGN 11
'˙' U+02D9 DOT ABOVE 8
'῾' U+1FFE GREEK DASIA 7
'*' U+002A ASTERISK 7
'&' U+0026 AMPERSAND 5
'/' U+002F SOLIDUS 4
'•' U+2022 BULLET 3
'῞' U+1FDE GREEK DASIA AND OXIA 2
'͵' U+0375 GREEK LOWER NUMERAL SIGN 1
'“' U+201C LEFT DOUBLE QUOTATION MARK 1
'[' U+005B LEFT SQUARE BRACKET 1


In [ ]:
def find_middle_dot_contexts(text):
    if not isinstance(text, str):
        return []

    # 1–30 non-space characters on each side of ·
    pattern = r'(\S{1,30})\s*·\s*(\S{1,30})'
    return re.findall(pattern, text)

all_middle_dot_splits = []

for text in df['Text_clean'].dropna():
    all_middle_dot_splits.extend(find_middle_dot_contexts(text))

print(f"Found {len(all_middle_dot_splits)} instances:\n")

for left, right in all_middle_dot_splits:
    print(f"{left} · {right}")

In [227]:
#corrections

pairs_to_join = [
    ('συν', 'ταχτικὲς'),
    ('κρατι', 'κὴ'),
    ('ἐκδηλώνον', 'ταν'),
    ('γλώσ', 'σας'),
    ('ἐπει', 'δὴ'),
    ('Σιάτι', 'στα'),
    ('μα', 'θήματα'),
    ('ἀρ', 'χαίων'),
    ('ἐπι', 'στημονική'),
    ('φι', 'λολόγων'),
    ('Ὄθω', 'μανοῖς'),
    ('Ρὴ', 'γα'),
    ('μα', 'θήσεων'),
    ('ἐπι', 'κίνδυνους'),
    ('ἀνα', 'σταλτική'),
    ('ἰ', 'διότητας'),
    ('ἀλ', 'λὰ'),
    ('ἀντι', 'δροῦσε'),
    ('᾿Απαρτί', 'ζονταν'),
    ('ἀτο', 'πία'),
    ('κάλα', 'μον'),
    ('ἔ', 'μαθαν'),
    ('ἐκεῖ', 'νη'),
    ('πί', 'στευε'),
    ('ἐπικίνδυ', 'νους'),
    ('ἀ', 'κόμα'),
    ('διατύπω', 'ναν'),
    ('γλῶσ', 'σαν'),
    ('ἀττι', 'κοὺς'),
    ('ἐπι', 'κράτηση'),
    ('πάν', 'τα'),
    ('ἀντε', 'θνικό'),
    ('ἀστει', 'εύομαι'),
    ('εὐρω', 'παϊκὸ'),
    ('ἐφημε', 'ρίδες'),
    ('ργάνω', 'ση'),
    ('αὐ', 'τοὺς'),
    ('ἔκα', 'νε'),
    ('ποί', 'ηση'),
    ('ἔ', 'θνος'),
    ('ὠφέλι', 'μες'),
    ('ἡ', 'μέραν'),
    ('που', 'λημένοι'),
    ('ἐθνι', 'κῆς'),
    ('Μι', 'στριώτης'),
    ('ἐπί', 'σημος'),
    ('ρα', 'γιάδες'),
    ('πῆ', 'γαν'),
    ('ἐπιδει', 'χθεῖσα'),
    ('γραμ', 'μένα'),
    ('Ἐλι', 'σαῖος'),
    ('ὑπέρτα', 'τον'),
    ('ἀπο', 'τελέσῃ'),
    ('ἔφκια', 'ναν'),
    ('ἐσω', 'τερικὴ'),
    ('ἐπαγ', 'γέλματα'),
    ('ἐκφραστι', 'κὸ'),
    ('ἔκ', 'θεση'),
    ('Παι', 'δαγωγική'),
    ('ἀνακρι', 'τικῆς'),
    ('Ἐπί', 'σης'),
    ('Ἐφτά', 'νησα'),
    ('ἐπίκρι', 'ση'),
    ('ἀκατόρθω', 'το'),
    ('κι', 'νήματα'),
    ('φι', 'λοσοφίας'),
    ('ἀρχαῖ', 'στῶν'),
    ('Ε', 'χει'),
]




In [228]:
def fix_verified_middle_dot_splits(text, pairs):
    if not isinstance(text, str):
        return text

    for left, right in pairs:
        pattern = (
            re.escape(left)
            + r'\s*·\s*'
            + re.escape(right)
        )
        replacement = left + right

        text = re.sub(pattern, replacement, text)

    return text

In [229]:
df['Text_clean'] = df['Text_clean'].apply(
    lambda x: fix_verified_middle_dot_splits(x, pairs_to_join)
)

In [ ]:
all_middle_dot_splits = []

for text in df['Text_clean'].dropna():
    all_middle_dot_splits.extend(find_middle_dot_contexts(text))

print(f"Found {len(all_middle_dot_splits)} instances:\n")

for left, right in all_middle_dot_splits:
    print(f"{left} · {right}")

In [ ]:
remaining = []

for page, text in zip(df['Page_Number'], df['Text_clean']):
    if not isinstance(text, str):
        continue

    for m in re.finditer(r'(\S+)\s*·\s*(\S+)', text):
        remaining.append(
            (page, m.group(1), m.group(2))
        )

print(f"Remaining: {len(remaining)}")

for page, left, right in remaining:
    print(page, f"{left} · {right}")

In [232]:
#final removal of middfle dot that does not split words
df['Text_clean'] = (
    df['Text_clean']
    .str.replace('·', ' ', regex=False)
)

In [4]:
# Display the df to check
df['Text_clean'][2]

### Remove special characters

In [234]:
print(df['Text_clean'].dtype)  
print(df['Text_clean'].apply(type).value_counts()) 

str
Text_clean
<class 'str'>    252
Name: count, dtype: int64


In [137]:
import re
def clean_text2(text):
    """
    Cleans text while preserving essential punctuation:
    - Keeps: `.`, `?`, `;`, `!`, `:`
    - Removes unwanted symbols like `•`, `#`, `♦`, `’`, etc.
    """
    if not isinstance(text, str):
        return text  # Return unchanged if not a string

    # Remove unwanted special characters (DO NOT remove `?`, `;`, `:`, `!`, or `.`)
    text = re.sub(r"[\r\n\xa0$•)%*■®῾£^¥᾽»'♦’#“`«»(/\]]", " ", text)  # No `?`, `;`, `:`, `!`, `.`

    # Normalize spaces (removes extra spaces)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [138]:
df['Text_clean'] = df['Text_clean'].map(lambda x: clean_text2(x))

In [235]:
#display the texts better
pd.set_option('display.width', 50)        
pd.set_option('display.max_colwidth', 800)

In [3]:
# Check
df[['Text_no_foot', 'Text_clean']].tail()

### Fix word spelling with frequency analysis

In [238]:
import nltk
nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize
import itertools
from collections import Counter


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
def tokenize(text):
    # Tokenize without lowercasing
    tokens = word_tokenize(text, language='greek')  # Specify Greek for correct tokenization
    #tokens = word_tokenize(text.lower(), language='greek')  # Specify Greek for correct tokenization
    return tokens


In [241]:
# Apply tokenization to our dataset
df['tokens'] = df['Text_clean'].apply(tokenize)

In [244]:
#  concatenate all token lists into a single list
all_tokens = list(itertools.chain.from_iterable(df['tokens']))

# Count word frequencies
word_counts = Counter(all_tokens)



1


In [2]:
# check most common words
print(word_counts.most_common(900)) 

In [ ]:
target = " και "

for _, row in df.iterrows():
    text = row["Text_clean"]

    if target in text:
        start = 0

        while True:
            pos = text.find(target, start)

            if pos == -1:
                break

            context = text[
                max(0, pos - 80):
                min(len(text), pos + len(target) + 80)
            ]

            print(f"\nPAGE {row['Page_Number']}")
            print(context)
            print("-" * 100)

            start = pos + len(target)

In [250]:
# spotted some mistakes...

corrections = {
    'ἄλ λης': 'ἄλλης',
    'αὐ τῆς': 'αὐτῆς',
    'γλωσ αικοῦ': 'γλωσσικοῦ',
    'δ πως': 'όπως',
    'δ μιλος': 'όμιλος',
    'δ Κονδυλάκης': 'ο Κονδυλάκης',
    'και ταστροφάς': 'καταστροφάς',
    'και ρὸς': 'καιρὸς',
    'νεοελ ληνικοῦ': 'νεοελληνικοῦ',
    'νεοελ ληνικό': 'νεοελληνικό',
    'γλωσσι κὸ': 'γλωσσικὸ',
    'γλωσσι κῆς': 'γλωσσικῆς',
    'γλωσσι κῆ': 'γλωσσικῆ',
    'γλωσσι κὸς': 'γλωσσικὸς',
    'ἀρι χή': 'ἀρχή',
    'μακραί νει': 'μακραίνει',
    'και θιερώθηκε': 'καθιερώθηκε',
    'και θὼς': 'καθὼς',
    'και θένα': 'καθένα',
    'και ταργήσει': 'καταργήσει',
    'και ραβοκυραίοι': 'καραβοκυραίοι',
    'και τόρθωσιν': 'κατόρθωσιν',
    'και ταφρόνεσις': 'καταφρόνεσις',
    'πνευ ματεμφόρων': 'πνευματεμφόρων',
    'διαιρε θῶσι': 'διαιρεθῶσι',
    'και νένας': 'κανένας',
    'και λογερικὲς': 'καλογερικὲς',
    'και ταλάβαιναν': 'καταλάβαιναν',
    'και λὸς': 'καλὸς',
    'και τηγορίες': 'κατηγορίες',
    'και θαρεύουσα': 'καθαρεύουσα',
    'και λαμαράδες': 'καλαμαράδες',
    'και θόλου': 'καθόλου',
    'και τατάραξαν': 'κατατάραξαν',
    'και νένα': 'κανένα',
    'και θυστερημένα': 'καθυστερημένα',
    'και θαρεύουσας': 'καθαρεύουσας',
    'και λύτερα': 'καλύτερα',
    'Εὐ αγγελικῆς': 'Εὐαγγελικῆς',
    'και θαρευουσιανισμοῦ': 'καθαρευουσιανισμοῦ',
    'αὐ τὴ': 'αὐτὴ',
    'και θιερωθεί': 'καθιερωθεί',
    'και βγάδες': 'καβγάδες',
    'και ταγίνονται': 'καταγίνονται',
    'και νεἰς': 'κανείς',
    'και τάσταση': 'κατάσταση',
    'ἀν ναστατώσεις': 'ἀναστατώσεις',
    'και ταφέρεται': 'καταφέρεται',
    'και θίσταται': 'καθίσταται',
    'καταφρο νοῦμε': 'καταφρονοῦμε',
    'και ταφρονοῦν': 'καταφρονοῦν',
    'και νόνες': 'κανόνες',
    'και ταντάει': 'καταντάει',
    'και θιερώθηκαν': 'καθιερώθηκαν',
    'και ταδιωχθῶσι': 'καταδιωχθῶσι',
    'και ταβιβάζουσι': 'καταβιβάζουσι',
    'και τάλαβε': 'κατάλαβε',
    'και τάφεραν': 'κατάφεραν',
    'κατηγο ρούμενου': 'κατηγορούμενου',
    'ὑ πουργοί': 'ὑπουργοί',
    'κατηγορούμε νου': 'κατηγορούμενου',
    'και θηγητής': 'καθηγητής',
    
}

for old, new in corrections.items():
    df["Text_clean"] = df["Text_clean"].str.replace(
        old,
        new,
        regex=False
    )

In [ ]:
# Lets check some more 
# concatenate all token lists into a single list
all_tokens = list(itertools.chain.from_iterable(df['tokens']))

# Count word frequencies
word_counts = Counter(all_tokens)

# Extract words with frequency less than 5
least_common_words = {word: freq for word, freq in word_counts.items() if freq < 5}

# Sort least common words from more frequent to less frequent
sorted_least_common = sorted(least_common_words.items(), key=lambda x: x[1], reverse=True)

# Print sorted least common words
print("Words with frequency less than 5 (sorted):")
for word, freq in sorted_least_common:
    print(f"{word}: {freq}")

In [264]:
# Export to confirm quotes
df.to_excel(
    "Kordatos_clean.xlsx",
    index=False
)

### Reimport after manual inspection

In [112]:
df = pd.read_excel(
    "Kordatos_clean_marked.xlsx"
)

# Mark quotes

In [113]:
def find_unmatched_quotes(df, text_col='Text_clean', page_col='Page_Number'):
    stack = []
    unmatched_closing = []

    for idx, row in df.iterrows():
        text = row[text_col]

        if not isinstance(text, str):
            continue

        page = row[page_col]

        for pos, char in enumerate(text):
            if char == '«':
                stack.append({
                    'page': page,
                    'index': idx,
                    'position': pos,
                    'text': text
                })

            elif char == '»':
                if stack:
                    stack.pop()
                else:
                    unmatched_closing.append({
                        'page': page,
                        'index': idx,
                        'position': pos,
                        'text': text
                    })

    return stack, unmatched_closing


unmatched_opening, unmatched_closing = find_unmatched_quotes(df)

In [114]:
print(f"Unmatched «: {len(unmatched_opening)}\n")

for item in unmatched_opening:
    text = item['text']
    pos = item['position']

    context = text[
        max(0, pos - 100):
        min(len(text), pos + 400)
    ]

    print(f"PAGE {item['page']}")
    print(context)
    print("-" * 100)

Unmatched «: 0



In [115]:
filtered_text = df[df['Page_Number'] == 51]
filtered_text

,Page_Number,Text_clean,word_count
44,51,"σχολεία δὲν ἔγιναν γιὰ τὸ Ψαλτήρι, οὔτε γιὰ τὴ γραμματι κή, μὰ γιὰ τὴν πλατειὰ μόρφωση, γιὰ τὸ φωτισμὸ τοῦ λαοῦ. Ἡ γραμματικὴ δὲν εἶναι ὁ σκοπός, μὰ τὸ μέσο. Καὶ γι αὐτὸ μάλιστα τὰ σχολικά βιβλία ἔπρεπε νὰ εἶναι γραμμένα σε ἁπλὴ γλώσσα, στὴ γλώσσα ποὺ μιλοῦσε καὶ καταλάβαινε ὁ λαός. Ὁ Μ. Γεωργίου τὸ βιβλίο του ᾿Αποθήκη τῶν παίδων, ποὺ βγήκε στὰ 1809, τὸ δικαιολογεῖ ἔτσι: «Τύπωσα τὴν Αποθήκη τῶν παίδων γιὰ νὰ μὴν παπαγαλλίζουν αμόνον οἱ μαθηταὶ τὴν Γραμματικήν τοῦ Λασκάρεως καὶ τὴν Ὀκτώηχον μ...",279


In [116]:
df. columns

Index(['Page_Number', 'Text_clean', 'word_count'], dtype='str')

In [117]:
df["marked_text"] = df["Text_clean"].copy()

## Join text pages per chapter

In [118]:
# Identify rows with NaN values in the 'text_no_poem' column
rows_with_nans = df[df['marked_text'].isnull()]

# Display the rows with NaN values
print(rows_with_nans)

    Page_Number Text_clean  word_count marked_text
17           24        NaN           0         NaN


In [119]:
df = df.dropna(subset=['marked_text']).reset_index(drop=True)

In [120]:
# Concatenate all sentences into a single text block
text_to_join = ' '.join(df['marked_text'])
#text_to_join

In [121]:
# Initialize the chapter markers in order
chapter_markers = [
    "ΠΡΟΛΟΓΟΣ",
    "ΕΙΣΑΓΩΓΗ",
    "ΚΕΦΑΛΑΙΟ ΠΡΩΤΟ",
    "ΚΕΦΑΛΑΙΟ ΔΕΥΤΕΡΟ",
    "ΚΕΦΑΛΑΙΟ ΤΡΙΤΟ",
    "ΚΕΦΑΛΑΙΟ ΤΕΤΑΡΤΟ",
    "ΚΕΦΑΛΑΙΟ ΠΕΜΠΤΟ",
    "ΚΕΦΑΛΑΙΟ ΕΚΤΟ",
    "ΚΕΦΑΛΑΙΟ ΕΒΔΟΜΟ",
    "ΚΕΦΑΛΑΙΟ ΟΓΔΟΟ",
    "ΚΕΦΑΛΑΙΟ ΕΝΑΤO",
    "ΠΙΝΑΚΑΣ ΟΝΟΜΑΤΩΝ"  # Last marker; content after this is ignored
]

In [122]:
# make sure the text starts from ΠΡΟΛΟΓῸΟΣ
start_index = text_to_join.find("ΠΡΟΛΟΓΟΣ")
if start_index == -1:
    raise ValueError("The word ΠΡΟΛΟΓΟΣ was not found in the text.")
text_to_join = text_to_join[start_index:]

# Split the text into chapters based on the markers
chapters = []
for i in range(len(chapter_markers) - 1):
    current_marker = chapter_markers[i]
    next_marker = chapter_markers[i + 1]
    
    # Find the content for the current chapter
    current_start = text_to_join.find(current_marker)
    next_start = text_to_join.find(next_marker)
    
    if current_start == -1:
        raise ValueError(f"The marker {current_marker} was not found in the text.")
    
    if next_start == -1:
        chapter_text = text_to_join[current_start + len(current_marker):].strip()
    else:
        chapter_text = text_to_join[current_start + len(current_marker):next_start].strip()
    
    # Append the chapter title and text to the list
    chapters.append({"Chapters": current_marker, "Text": chapter_text})



In [123]:
# Create a df
df1 = pd.DataFrame(chapters)

In [124]:
#display the texts better
pd.set_option('display.width', 50)        
pd.set_option('display.max_colwidth', 500)

In [125]:
df1

,Chapters,Text
0,ΠΡΟΛΟΓΟΣ,"Μὴ ζητᾶς στὰ λατινικὰ πῶς θὰ γράψεις καλὰ γερμανικά. Ρώτα τη μάνα σου στο σπίτι, τὰ παιδιὰ στοὺς δρόμους, τοὺς ἀνθρώπους τοῦ λαοῦ στὴν ἀγορά. Κοίταζέ τους στο στόμα πῶς μιλάνε καὶ ἔτσι γράφε. ΛΟΥΘΗΡΟΣ Τὸ γλωσσικό ζήτημα πᾶνε ἑκατὸ καὶ παραπάνω χρόνια ποὺ κάθε τόσο συνταράζει τὴν ἑλληνικὴ κοινωνία. Προδό τες, μαλλιαροί, ἄθεοι, εἶναι τὰ ἐπίθετα ποὺ κολλάει ὁ ἐπιστημονικὸς ὄχλος στοὺς ὀπαδοὺς τῆς λαϊκῆς γλώσσας. Μερικοὶ θὰ νομίζουν πὼς μοναχὰ στὶς ἡμέρες μας ὁ γλωσσικὸς ἀγώνας πῆρε τόση ὀξύτητα..."
1,ΕΙΣΑΓΩΓΗ,"Ἡ γλώσσα εἶναι ἕνα ἐργαλεῖο μὲ τὸ ὁποῖο οἱ ἄνθρωποι συνεννοοῦνται. Τὸ ἐργαλεῖο ὅμως αὐτὸ δὲν εἶναι ἄφωνο ὅπως τὰ κοινὰ ἐργαλεῖα, ἀλλὰ φωνητικό, γιατὶ ἡ γλώσσα δὲν εἶναι τίποτε ἄλλο παρὰ ἀθροισμα ἀπὸ συναρμολογημένες φωνές. Καμιὰ λοιπὸν γλώσσα δὲν εἶναι οὔτε ἀριστοκρατική οὔτε χυ δαία. Ὑπάρχουν μόνο γλῶσσες φτωχές, ἀνεξέλιχτες καὶ γλώσσες τελειοποιημένες, μὲ πλούσιο λεξιλόγιο καὶ ἐκφρα στικὸ ὑλικό. Ἡ γλωσσολογία ἔχει διαπιστώσει πὼς ὁ κυριό τερος παράγοντας ποὺ κάνει μιὰ γλώσσα πλούσια εἶναι ..."
2,ΚΕΦΑΛΑΙΟ ΠΡΩΤΟ,"Τὸ γλωσσικό ζήτημα στὸν καιρὸ τῆς τουρκοκρατίας ( 1453-1821 ) Η διγλωσσία στο Βυζάντιο ᾿Απ᾿ ὅσα ἔχουμε πεῖ ἴσαμε δῶ, ξέρουμε πὼς διγλωσσία ὑπῆρ χε καὶ πρὶν τοῦ χριστιανισμοῦ. Ἐνῶ ὅμως τὰ βιβλία τῆς Νέας Διαθήκης γράφτηκαν στη λαϊκή γλώσσα τῆς τοτινῆς ἐποχῆς, ποὺ ἦταν κοινὴ ἀνάμεσα στὰ λαϊκὰ ἑλληνικὰ καὶ ἑλληνόφωνα στρώματα ποὺ κατοικούσαν ὄχι μόνο στὴν κυν ρίως Ἑλλάδα μὰ καὶ στὴ Μικρασία, Αἴγυπτο καὶ σὲ πολλὲς ἀκόμα πόλεις τῆς Ἰταλίας, οἱ πατέρες τῆς ἐκκλησίας καὶ γενικὰ οἱ θεωρητικοὶ καὶ οἱ..."
3,ΚΕΦΑΛΑΙΟ ΔΕΥΤΕΡΟ,"Τὸ μεσουράνημα τοῦ ἀττικισμοῦ ( Περίοδος 1821-1830 ) Τὸ γλωσσικό ζήτημα στὸ Εἰκοσιένα Όταν ξέσπασε ἡ Ἐπανάσταση, ἡ γλωσσική πάλη παραμε ρίστηκε. Ωστόσο όμως κρυφόβραζε τὸ πάθος ἁνάμεσα στὰ δυὸ κόμματα. Στὰ πρῶτα χρόνια οἱ Κοραῖστὲς ἐπικρατοῦσαν. Λίγο πολύ, προκηρύξεις, δημόσια ἀλληλογραφία, που λιτεύματα κλπ., εἶναι γραμμένα σὲ ἁπλὴ ὁπωσδήποτε γλώσ σα, χωρίς βέβαια καμιὰ ὁμοιομορφία στὸ τυπικὸ καὶ τὴ γραμματική. Οσοι ἦταν γραμματισμένοι ἀστοί, γράφανε ἁν πλά. Οἱ Φαναριώτες ὅμως ἐξακολουθούσα..."
4,ΚΕΦΑΛΑΙΟ ΤΡΙΤΟ,"Τὸ κήρυγμα τοῦ Ψυχάρη καὶ ἡ δύση τοῦ ἀττικισμοῦ. Νέοι γλωσσικοὶ ἀγῶνες Ἐνῶ οἱ ἀττικιστὲς πίστευαν πὼς ὁ ἑλληνικὸς λαὸς μέσα σὲ λίγα χρόνια θὰ μάθαινε νὰ κουβεντιάζει ὅπως καὶ οἱ ἀρι χαῖοι, ἡ γλωσσικὴ ἐξέλιξη τραβούσε πρὸς τὴ δημοτική σ 8λες τὶς ἐκδηλώσεις τῆς πνευματικῆς ζωῆς τοῦ ἔθνους. Ηταν πολλοὶ τώρα ποὺ δὲν πίστευαν πὼς ἦταν δυνατὸ νὰ ξαναγυρίσουμε στὸ γλωσσικό καθεστώς τῆς ἀρχαίας καὶ παι ραδέχονταν πὼς οἱ γνώμες τοῦ Κοραή ἦταν οἱ πιὸ σωστές. Ωστόσο ὑπῆρχαν καὶ ἄλλοι ποὺ ἦταν πιὸ ριζοσ..."
5,ΚΕΦΑΛΑΙΟ ΤΕΤΑΡΤΟ,"Τὸ ἅπλωμα τοῦ δημοτικισμοῦ. Δημοτικὴ καὶ φιλολογία. Ὕστερ᾿ ἀπὸ τὰ Εὐαγγελικὰ καὶ τὰ Ὀρεστειακά, ἡ γλωσσι κὴ ἀντίδραση προσπάθησε, μὲ ὅπλο τὴ συκοφαντία, νὰ ἐμποδίσει τὸ ἁπλωμα τοῦ δημοτικισμοῦ. Δὲν τὸ κατάφερε δι μως. Ὅλοι οἱ νέοι λογοτέχνες ( ποιητὲς καὶ πεζογράφοι ), ποὺ παραμερίζουν τους παλιούς καὶ ἀνεβαίνουν στὸ νεοελληνικό Παρνασσό, εἶναι δημοτικιστὲς καὶ ἀνάμεσα σ αὐ τοὺς στέκεται ὁ Κωστής Παλαμᾶς, ποὺ ἔγινε ὁ στυλοβάτης τοῦ δημοτικισμοῦ μὲ τὸ κύρος καὶ τὴν πλούσια ποιητική του παραγω..."
6,ΚΕΦΑΛΑΙΟ ΠΕΜΠΤΟ,"Τὰ ἀθεϊκὰ τοῦ Βόλου. Ο Βόλος, κέντρο πολιτικοκοινωνικών ζυμώσεων στὰ χρόνια 1905-1910 Ὁ Βόλος ἀπὸ τὴ γεωγραφική του θέση εἶχε γίνει σπουδαίο ἐμποροβιομηχανικό κέντρο στὰ ὕστερα ἀπὸ τὸ 1900 χρόνια καὶ οὐσιαστικὰ κρατοῦσε τὴν οἰκονομικὴ ζωὴ ὅλης τῆς Θεσσαλίας. Γι᾿ αὐτὸ μέσα στὴν πόλη αὐτὴ ἀκούστηκαν ἔντονες οἱ πρῶτες κραυγές τοῦ ἀφυπνιζόμενου ἀγροτισμοῦ καὶ ἐργατισμού. Παρ ὅλη τὴ δύναμη τῶν παλαιῶν κομμάτων, δὲν ἄργησαν νὰ παρουσιαστοῦν καὶ λαϊκὲς ὁμάδες, ποὺ ἔρριχναν καινούργια συνθήματα πολι..."
7,ΚΕΦΑΛΑΙΟ ΕΚΤΟ,"Η γλωσσοεκπαιδευτική μεταρρύθμιση. Ἡ εἰσαγωγὴ τῆς δημοτικῆς στὰ δημοτικά σχολεία Τὸ κίνημα τῆς Θεσσαλονίκης ( 1917 ) ἦταν ἕνα κίνημα ἀστικὸ μὲ ἰμπεριαλιστικές τάσεις. Μὰ στὸ βάθος του ἦταν συμπλήρωμα τοῦ κινήματος τοῦ Γουδιού ( 1909 ). Ἐπειδὴ τὰ φεουδαρχικὰ στοιχεῖα στάθηκαν δισταχτικά, ἡ πλειοψηφία τῆς ἀστικῆς τάξης ( κυρίως τὸ μεγάλο κεφά

In [126]:
# Dictionary to correct chapters

replacements = {
     "ΠΡΟΛΟΓΟΣ": "Preface",
    "ΕΙΣΑΓΩΓΗ": "Introduction",
    "ΚΕΦΑΛΑΙΟ ΠΡΩΤΟ": "Chapter_1",
    "ΚΕΦΑΛΑΙΟ ΔΕΥΤΕΡΟ": "Chapter_2",
    "ΚΕΦΑΛΑΙΟ ΤΡΙΤΟ": "Chapter_3",
    "ΚΕΦΑΛΑΙΟ ΤΕΤΑΡΤΟ": "Chapter_4",
    "ΚΕΦΑΛΑΙΟ ΠΕΜΠΤΟ": "Chapter_5",
    "ΚΕΦΑΛΑΙΟ ΕΚΤΟ": "Chapter_6",
    "ΚΕΦΑΛΑΙΟ ΕΒΔΟΜΟ": "Chapter_7",
    "ΚΕΦΑΛΑΙΟ ΟΓΔΟΟ": "Chapter_8",
    "ΚΕΦΑΛΑΙΟ ΕΝΑΤO": "Chapter_9",

}
    
# replace words based on the designated dictionary
def replace_words(text, replacements):
    for incorrect, correct in replacements.items():
        text = text.replace(incorrect, correct)
    return text


In [127]:
df1['Chapters'] = df1['Chapters'].apply(lambda x: replace_words(str(x), replacements))

In [ ]:
#check
df1

In [129]:
#Make sure we have strings
df1['Text'] = df1['Text'].apply(lambda x: str(x) if not isinstance(x, str) else x)

### Separate quoted text from authorial text

In [133]:
import re

def extract_quoted_text(text):
    if not isinstance(text, str):
        return ""

    quotes = re.findall(
        r"«.*?»",
        text,
        flags=re.DOTALL
    )

    return " ".join(q.strip() for q in quotes)

def remove_quoted_text(text):
    if not isinstance(text, str):
        return ""

    text_no_quote = re.sub(
        r"«.*?»",
        " ",
        text,
        flags=re.DOTALL
    )

    text_no_quote = re.sub(r"\s+", " ", text_no_quote)

    return text_no_quote.strip()

In [134]:
df1["quoted_text"] = df1["Text"].apply(extract_quoted_text)

df1["text_no_quote"] = df1["Text"].apply(remove_quoted_text)

In [ ]:
df1

### Lets count words

In [139]:
import re

def count_words(text):
    if not isinstance(text, str):
        return 0

    words = re.findall(
        r"[^\W\d_]+(?:['’᾿᾽][^\W\d_]*)?",
        text,
        flags=re.UNICODE
    )

    return len(words)

In [140]:
df1["Original_Text_words"] = df1["Text"].apply(count_words)
df1["text_no_quote_words"] = df1["text_no_quote"].apply(count_words)
df1["quoted_text_words"] = df1["quoted_text"].apply(count_words)

In [141]:
df1.columns

Index(['Chapters', 'Text', 'text_no_quote', 'quoted_text', 'Original_Text_words', 'text_no_quote_words', 'quoted_text_words'], dtype='str')

In [142]:
df1["check"] = (
    df1["text_no_quote_words"]
    + df1["quoted_text_words"]
    - df1["Original_Text_words"]
)

df1["check"].value_counts()

check
0    11
Name: count, dtype: int64

In [143]:
total_words = df1["Original_Text_words"].sum()
total_text_no_quote_words = df1["text_no_quote_words"].sum()
total_quoted_text_words = df1["quoted_text_words"].sum()



In [144]:
print(f"TOTAL WORD COUNT: {total_words:,}")
print(f"NO QUOTES TEXT WORD COUNT: {total_text_no_quote_words:,}")
print(f"QUOTED TEXT WORD COUNT: {total_quoted_text_words:,}")

TOTAL WORD COUNT: 58,769
NO QUOTES TEXT WORD COUNT: 41,694
QUOTED TEXT WORD COUNT: 17,075


In [148]:
print("Own %:", total_text_no_quote_words / total_words * 100)
print("Quoted %:", total_quoted_text_words / total_words * 100)

Own %: 70.94556654018275
Quoted %: 29.054433459817254


## Begin cleaning

In [21]:
# confirm we have strings
df1['Text'] = df1['Text'].apply(lambda x: str(x) if not isinstance(x, str) else x)

In [22]:
import re
import unicodedata

def conservative_clean(text):

    if not isinstance(text, str):
        return ""

    # 1. unicode normalization
    # NFC preserves polytonic Greek characters and « » quotation marks
    text = unicodedata.normalize("NFC", text)

    # 2. replace non-breaking spaces with normal spaces
    text = text.replace("\u00A0", " ")

    # 3. replace tabs with spaces
    text = text.replace("\t", " ")

    # 4. replace multiple spaces with a single space
    # does NOT alter punctuation or quotation marks
    text = re.sub(r"[ ]{2,}", " ", text)

    # 5. remove spaces BEFORE basic punctuation
    # e.g. "word ." -> "word."
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)

    # 6. remove spaces immediately inside Greek quotation marks
    # « text » -> «text»
    text = re.sub(r"«\s+", "«", text)
    text = re.sub(r"\s+»", "»", text)

    # 7. normalize multiple blank lines
    text = re.sub(r"\n[ \t]*\n+", "\n", text)

    return text.strip()


In [23]:
df1['text_no_quote_clean'] = df1['text_no_quote'].apply(conservative_clean)
df1['Text_clean'] = df1['Text'].apply(conservative_clean)

### Check for that middle dot again

In [180]:
suspects = []

patterns = {
    "middle_dot_between_letters": r"[^\W\d_]\s*·\s*[^\W\d_]",
    "double_middle_dot": r"··+",
    "space_before_punctuation": r"\s+[.,;:!?]",
    "non_greek_arabic": r"[\u0600-\u06FF]",
}

for _, row in df1.iterrows():

    text = row["Text"]

    for label, pattern in patterns.items():

        for m in re.finditer(pattern, text):

            suspects.append({
                "Chapter": row["Chapters"],
                "Type": label,
                "Instance": m.group(),
                "Context": text[
                    max(0, m.start()-70):
                    min(len(text), m.end()+70)
                ]
            })

suspect_df = pd.DataFrame(suspects)


In [182]:
suspect_df

### Convert to monotonic

In [24]:
import unicodedata
import re

def convert_polytonic_to_simple(text):
    """
    Removes (δασεία and ψιλή) and converts περισπωμένη to oξεία.
    
    Args:
        text (str): Polytonic Greek text.
        
    Returns:
        str: Simplified text with πνευματα removed and περισπωμένη converted to ὀξεία.
    """
    # Normalize the text to NFD (decomposed form)
    text = unicodedata.normalize('NFD', text)
    
    # Remove breathing marks: δασεία (◌̔) and ψιλή (◌̓)
    text = re.sub(r'[\u0313\u0314]', '', text)  # Removes U+0313 (ψιλή) and U+0314 (δασεία)
    
    # Replace περισπωμένη (◌̃) with ὀξεία (◌́)
    text = re.sub(r'\u0342', '\u0301', text)  # Replace U+0342 (περισπωμένη) with U+0301 (ὀξεία)
    
    # Normalize back to NFC (composed form)
    text = unicodedata.normalize('NFC', text)
    
    return text

In [25]:
# Apply the function to simplify polytonic Greek
df1['text_monotonic'] = df1['Text_clean'].apply(convert_polytonic_to_simple)
df1['text_no_quote_monotonic'] = df1['text_no_quote_clean'].apply(convert_polytonic_to_simple)



#### We have some βαρείες left... Lets convert them to οξειες

In [30]:
def convert_polytonic_to_monotonic(text):
    """
    Convert Greek text from polytonic to monotonic orthography.
    Handle characters like \u1fc4 and replace with monotonic equivalents.
    """
    # Normalize to decomposed form (NFD)
    text = unicodedata.normalize('NFD', text)
    
    # Replace \u1fc4 (ᾴ) with άι
    text = text.replace('\u1fc4', 'ά')
    
    # Remove breathing marks: δασεία (◌̔) and ψιλή (◌̓)
    text = re.sub(r'[\u0313\u0314]', '', text)
    
    # Replace grave accent (◌̀) with acute accent (◌́)
    text = text.replace('\u0300', '\u0301')
    
    # Normalize back to composed form (NFC)
    text = unicodedata.normalize('NFC', text)
    
    return text

In [31]:
#just making sure we have strings...
df1['text_monotonic'] = df1['text_monotonic'].apply(lambda x: str(x) if not isinstance(x, str) else x)
df1['text_no_quote_monotonic'] = df1['text_no_quote_monotonic'].apply(lambda x: str(x) if not isinstance(x, str) else x)

In [33]:
#Apply the function to simplify polytonic Greek
df1['text_monotonic'] = df1['text_monotonic'].apply(convert_polytonic_to_monotonic)

In [34]:
df1['text_no_quote_monotonic'] = df1['text_no_quote_monotonic'].apply(convert_polytonic_to_monotonic)

### Clean ypogegrameni

In [35]:
import unicodedata

# Function to remove the iota subscript
def remove_iota_subscript(text):
    # Decompose the Unicode characters into base and diacritics
    decomposed = unicodedata.normalize("NFD", text)
    # Remove the iota subscript (U+0345)
    cleaned = ''.join(char for char in decomposed if unicodedata.name(char) != "COMBINING GREEK YPOGEGRAMMENI")
    # Recompose the text into normal form
    return unicodedata.normalize("NFC", cleaned)

In [36]:
# Apply to the DataFrame column
df1['text_monotonic'] = df1['text_monotonic'].apply(remove_iota_subscript)

In [37]:
df1['text_no_quote_monotonic'] = df1['text_no_quote_monotonic'].apply(remove_iota_subscript)

### Fix word spelling with frequency analysis

In [6]:
from nltk.tokenize import word_tokenize
import itertools
from collections import Counter


In [ ]:
def tokenize(text):
    # Tokenize without lowercasing
    tokens = word_tokenize(text, language='greek')  # Specify Greek for correct tokenization
    #tokens = word_tokenize(text.lower(), language='greek')  # Specify Greek for correct tokenization
    return tokens


In [9]:
# Apply tokenization to our dataset
df1['tokens'] = df1['text_monotonic'].apply(tokenize)

In [16]:
#  concatenate all token lists into a single list
all_tokens = list(itertools.chain.from_iterable(df1['tokens']))

# Count word frequencies
word_counts = Counter(all_tokens)

# Extract words with frequency less than 8
least_common_words = {word: freq for word, freq in word_counts.items() if freq < 8}

# Sort least common words from more frequent to less frequent
sorted_least_common = sorted(least_common_words.items(), key=lambda x: x[1], reverse=True)


In [ ]:
# Inspect the most frequent tokens after constructing the counter
print(word_counts.most_common(900))


In [ ]:
# Print sorted least common words
print("Words with frequency less than 8 (sorted):")
for word, freq in sorted_least_common:
    print(f"{word}: {freq}")

### Check 'broken' words

In [84]:
def get_partial_context(text, target, window=5):
    """
    Finds the context around a target substring in the text, 
    including cases where words are split.

    Parameters:
    text (str): The text to search.
    target (str): The specific substring to find (partial match).
    window (int): The number of words before and after the target.

    Returns:
    list: A list of contexts where the target substring appears.
    """
    # Regex to match the target substring and surrounding words
    pattern = rf'((?:\S+\s+){{0,{window}}})(\S*{re.escape(target)}\S*)(?:\s+(\S+)){{0,{window}}}'
    
    # Find all matches
    matches = re.finditer(pattern, text)
    
    # Collect the contexts
    contexts = []
    for match in matches:
        before = match.group(1).strip()  # Words before the target
        target_word = match.group(2).strip()  # Word containing the target
        after = match.group(3) or ''  # Words after the target (if any)
        contexts.append(f"{before} {target_word} {after}".strip())
    
    return contexts

# Example 
text = "Έξω απ αυτή χτίζουμε καί θά χτί ζουμε στήν άμμο ."

# Specify the target substring
target = "χτί "

# Get the contexts for the target
contexts = get_partial_context(text, target, window=10)

# Print the results
print("Contexts found:")
for context in contexts:
    print(context)

Contexts found:
Έξω απ αυτή χτίζουμε καί θά χτί ζουμε .


In [145]:
# Specify the target string
target = " κών."

# Collect all contexts across the DataFrame
all_contexts = []
for text in df1['text_monotonic']:
    contexts = get_partial_context(text, target, window=20)
    all_contexts.extend(contexts)

# Print the results in a clean list format
print("Contexts found:")
for context in all_contexts:
    print(context)

Contexts found:


In [40]:
replacements = {
    r'\bχυ δαία\b': 'χυδαία',
    r'\bΛογι ωτατισμός\b': 'Λογιωτατισμός',
    r'\bτόν σο\b': 'τόσο',
    r'\b᾿Α ναγέννηση\b': 'Αναγέννηση',
    r'\bνεκραναστή σει\b': 'νεκραναστήσει',
    r'\bκλασσι κών\b': 'κλασσικών',
    r'\bσχέ σεων\b': 'σχέσεων',
    r'\bφεουδαρχι σμού\b': 'φεουδαρχισμού',
    r'\bδιομηχανικά\b': 'βιομηχανικά',
    r'\bτραγου διών\b': 'τραγουδιών',
    r'\bΚαι βαλιώτη\b': 'Καβαλιώτη',
    r'\bνεωτερι στής\b': 'νεωτεριστής',
    r'\bκαθιερω θεί\b': 'καθιερωθεί',
    r'\bκαθαρεύ ουσαςν\b': 'καθαρεύουσας',
    r'\bθρη σκείας\b': 'θρησκείας',
    r'\bθεμε λιακό\b': 'θεμελιακό',
    r'\bΚαι ταλάβαιναν\b': 'Καταλάβαιναν',
    r'\bλοι πόν\b': 'λοιπόν',
    r'\bαυ τούς\b': 'αυτούς',
    r'\bδάλι λο\b': 'άλλο',
    r'\bγραμματι σμένοι\b': 'γραμματισμένοι',
    r'\bχυν δαϊκά\b': 'χυδαϊκά',
    r'\bπολεμι κή\b': 'πολεμική',
    r'\bελληνιστι κή\b': 'ελληνιστική',
    r'\bλυρι κή\b': 'λυρική',
    r'\bεκκλησιαστι κή\b': 'εκκλησιαστική',
    r'\bγλωσ σικός\b': 'γλωσσικός',
    r'\bεπαναστατι κή\b': 'επαναστατική',
    r'\bγλωσσι κή\b': 'γλωσσική',
    r'\bκοινωνι κών\b': 'κοινωνικών',
    r'\bπαραμε ρίστηκε\b': 'παραμερίστηκε',
    r'\bγλώσ σα\b': 'γλώσσα',
    r'\bαν πλά\b': 'απλά',
    r'\bγλωσ σικό\b': 'γλωσσικό',
    r'\bΕλλά δας\b': 'Ελλάδας',
    r'\bλο γιωτάτων\b': 'λογιωτάτων',
    r'\bδάσκα λοι\b': "δάσκαλοι",
    r'\bλογιων τατισμού\b': "λογιωτατισμού",
     r'\bατι τικισμό\b': "αττικισμό",
     r'\bδι ποστηρίζουν\b': "υποστηρίζουν",
     r'\bσύν στημα\b': "σύστημα",
     r'\bσύνο ρα\b': "σύνορα",
     r'\bσυζητή σει\b': "συζητήσει",
     r'\bγυα λιά\b': "γυαλιά",
    r'\bδασκά λους\b': "δασκάλους",
    r'\bρου τινιέρικο\b': "ρουτινιέρικο",
    r'\bαντίδρα σης\b': "αντίδρασης",
    r'\bδι νομα\b': "όνομα",
    r'\bαυν ξάνουν\b': "αυξάνουν",
    r'\bευαγγέ λια\b': "ευαγγέλια",
    r'\bΚυβερνήσεως\.': "Κυβερνήσεως,",
    r'\b1\.000\b': '1000',
     r'\bδη μοτική\b': 'δημοτική',
    r'\bδη μοσιεύτηκε\b': 'δημοσιεύτηκε',
    r'\bπρόν\b': 'πριν',
    r'\bΚωνσταν τινίδη\b': 'Κωνσταντινίδη',
    r'\bσυνείδη ση\b': 'συνείδηση',
    r'\bθρησκευ τικό\b': 'θρησκευτικό',
    r'\bσχε τικό\b': 'σχετικό',
    r'\bπραγμα τικότητας\b': 'πραγματικότητας',
    r'\bζων τικό\b': 'ζωτικό',
    r'\bλδ γος\b': 'λόγος',
    r'\bπαράΈλεψε\b': 'παρέλειψε',
    r'\bαν ναρχία\b': 'αναρχία',
    r'\bδύσ κολη\b': 'δύσκολη',
    r'\bα κουστούν\b': 'ακουστούν',
    r'\bελληνι στές\b': 'ελληνιστές',
    r'\bδημοτικι στές\b': 'δημοτικιστές',
    r'\bεπιχειρη ματολογία\b': 'επιχειρηματολογία',
    r'\bαπο δοτικότητα\b': 'αποδοτικότητα',
    r'\bδρολογία\b': 'ορολογία',
    r'\bξα ναμορφωτής\b': 'ξαναμορφωτής',
    r'\bαλλη λοεπηρεάζονται\b': 'αλληλοεπηρεάζονται',
    r'\bυι ος\b': 'υιός',
    r'\bΙΙ\.χ\.\b': 'Π.χ.',
    r'\bπνεύμα τα\b': 'πνεύματα',
    r'\bαντεθνι κό\b': 'αντεθνικό',
    r'\bπαλαιοημερολο γιτών\b': 'παλαιοημερολογιτών',
    r'\bγύ ρευε\b': 'γύρευε',
    r'\bκαθαρευου σιάνος\b': 'καθαρευουσιάνος',
    r'\bαν τομικότητα\b': 'ατομικότητα',
    r'\bΙστο ρίας\b': 'Ιστορίας',
    r'\bπλου τοκρατίας\b': 'πλουτοκρατίας',
    r'\bστρώ ματα\b': 'στρώματα',
    r'\bπρο οδευτική\b': 'προοδευτική',
    r'\bπρό γραμμα\b': 'πρόγραμμα',
    r'\bπου λίτες\b': 'πολίτες',
    r'\bλό γου\b': 'λόγου',
    r"\bμορ'φή\b": "μορφή",
    r"\bδι μως\b": "όμως",
    r'\bΕίναι\.': 'Είναι',
    r'\bη\.': 'η',
    r'\bΤόμαν\.': 'Τόμαν',

   
}

In [41]:
def replace_words2(text, replacements):
    for incorrect, correct in replacements.items():
        text = re.sub(incorrect, correct, text)
    return text

In [42]:
# Apply the replacements to the columns
df1['text_monotonic'] = df1['text_monotonic'].apply(lambda x: replace_words2(str(x), replacements))

In [43]:
df1['text_no_quote_monotonic'] = df1['text_no_quote_monotonic'].apply(lambda x: replace_words2(str(x), replacements))

## Remove accents from stopwords
#### Targeted normalization of frequent forms
The following dictionary applies targeted, case-sensitive corrections to selected frequent forms identified during corpus inspection. 


In [44]:
pre_correction_dict = {
    'καί': 'και',
    'τού': 'του',
    'τό': 'το',
    'τής': 'της',
    'νά': 'να',
    'τήν': 'την',
    'τά': 'τα',
    'πού': 'που',
    'τούς': 'τους',
    'μέ': 'με',
    'δέν': 'δεν',
    'τών': 'των',
    'τή': 'τη',
    'γιά': 'για',
    'τόν': 'τον',
    'πώς': 'πως',
    'καί': 'και',
    'στό': 'στο',
    'στήν': 'στην',
    'τίς': 'τις',
    'στά': 'στα',
    'στή': 'στη',
    'μιά': 'μια',
    'Ό': 'Ο',
    'σέ': 'σε',
    'Ή': 'Η',
    'θά': 'θα',
    'δέ': 'δε',
    'μά': 'μα',
    'Τό': 'Το',
    'άν': 'αν',
    'Μά': 'Μα',
    'πιό': 'πιο',
    'Καί': 'Και',
    'τάς': 'τας',
    'στίς': 'στις',
    'στόν': 'στον',
    'στούς': 'στους',
    'πρός': 'προς',
    'σάν': 'σαν',
    'δυό': 'δυο',
    'Δέν': 'Δεν',
    'μή': 'μη',
    'Νά': 'Να',
    'μαζύ': 'μαζί',
    'Στήν': 'Στην',
    'σά': 'σα',
    'μήν': 'μην',
    'Σέ': 'Σε',
    'Στά': 'Στα',
    'πρίν': 'πριν',
    'τῴ': 'τω',
    'άς': 'ας',
    'μέν': 'μεν',
    'Τήν': 'Την',
    'Μέ': 'Με',
    'Άν': 'Αν',
    'τῄ': 'τη',
    'ό': 'ο',
    'είς': 'εις',
    'πρό': 'προ',
    'Μιά': 'Μια',
    'μιάς': 'μιας',
    'Γιά': 'Για',
    'ού': 'ου',
    'τοίς': 'τοις',
    'Στή': 'Στη',
    'τές': 'τες',
    'ή': 'η'
}

In [50]:
def pre_correct_stopwords_safe_case_sensitive(text, correction_dict):
    """
    Replaces stopwords with de-accented versions, preserving case.
    """
    # Compile a pattern for word boundaries and dictionary keys
    pattern = re.compile(r'\b(' + '|'.join(map(re.escape, correction_dict.keys())) + r')\b')
    
    def replace_with_case(match):
        """
        Replaces the matched word with the corresponding de-accented version,
        preserving the original case.
        """
        original_word = match.group(0)
        replacement_word = correction_dict[original_word] # Get the de-accented version

        # Check if the original word was all uppercase or title case
        if original_word.isupper():
            return replacement_word.upper()  # Replace with uppercase
        elif original_word.istitle():
            return replacement_word.title()  # Replace with title case
        else:
            return replacement_word  # Replace with lowercase

    # Replace using the custom replacement function
    return pattern.sub(replace_with_case, text)


In [46]:
# Apply to the whole DataFrame
df1['text_monotonic'] = df1['text_monotonic'].apply(lambda x: pre_correct_stopwords_safe_case_sensitive(x, pre_correction_dict))
df1['text_no_quote_monotonic'] = df1['text_no_quote_monotonic'].apply(lambda x: pre_correct_stopwords_safe_case_sensitive(x, pre_correction_dict))



In [65]:
# beter be safe than sorry
df1.to_csv('../kordatos_Preprocessed.csv', index=False, encoding='utf-8-sig')